In [1]:
import pandas
from survtrace.model import SurvTraceSingle
from survtrace.train_utils import Trainer
from survtrace.config import STConfig
from easydict import EasyDict

In [2]:
STConfig

{'data': 'metabric',
 'num_durations': 5,
 'horizons': [0.25, 0.5, 0.75],
 'seed': 1234,
 'checkpoint': './checkpoints/survtrace.pt',
 'vocab_size': 8,
 'hidden_size': 16,
 'intermediate_size': 64,
 'num_hidden_layers': 3,
 'num_attention_heads': 2,
 'hidden_dropout_prob': 0.0,
 'num_feature': 9,
 'num_numerical_feature': 5,
 'num_categorical_feature': 4,
 'out_feature': 3,
 'num_event': 1,
 'hidden_act': 'gelu',
 'attention_probs_dropout_prob': 0.1,
 'early_stop_patience': 5,
 'initializer_range': 0.001,
 'layer_norm_eps': 1e-12,
 'max_position_embeddings': 512,
 'chunk_size_feed_forward': 0,
 'output_attentions': False,
 'output_hidden_states': False,
 'tie_word_embeddings': True,
 'pruned_heads': {}}

In [9]:
from survtrace.dataset import load_data
# load data
STConfig2 = EasyDict(STConfig.copy())
df, df_train, df_y_train, df_test, df_y_test, df_val, df_y_val = load_data(STConfig2)

b:\Python\surv_mlp\experiments\models\survtrace\utils.py:78: UserWarning: Got event/censoring at start time. Should be removed! It is set s.t. it has no contribution to loss.
  warnings.warn("""Got event/censoring at start time. Should be removed! It is set s.t. it has no contribution to loss.""")


In [13]:
STConfig2

{'data': 'metabric',
 'num_durations': 5,
 'horizons': [0.25, 0.5, 0.75],
 'seed': 1234,
 'checkpoint': './checkpoints/survtrace.pt',
 'vocab_size': 8,
 'hidden_size': 16,
 'intermediate_size': 64,
 'num_hidden_layers': 3,
 'num_attention_heads': 2,
 'hidden_dropout_prob': 0.0,
 'num_feature': 9,
 'num_numerical_feature': 5,
 'num_categorical_feature': 4,
 'out_feature': 4,
 'num_event': 1,
 'hidden_act': 'gelu',
 'attention_probs_dropout_prob': 0.1,
 'early_stop_patience': 5,
 'initializer_range': 0.001,
 'layer_norm_eps': 1e-12,
 'max_position_embeddings': 512,
 'chunk_size_feed_forward': 0,
 'output_attentions': False,
 'output_hidden_states': False,
 'tie_word_embeddings': True,
 'pruned_heads': {},
 'labtrans': <survtrace.utils.LabelTransform at 0x1786830b750>,
 'duration_index': array([  0.        ,  42.68333435,  85.8666687 , 145.33333588,
        355.20001221])}

In [4]:
for k in STConfig:
    del STConfig2[k]
STConfig2

{'labtrans': <survtrace.utils.LabelTransform at 0x17826b7b380>,
 'duration_index': array([  0.        ,  42.68333435,  85.8666687 , 145.33333588,
        355.20001221])}

In [5]:
df_y_train.head()

,duration,event,proportion
0,2,0.0,0.226457
1,2,1.0,0.165919
6,3,0.0,0.552097
10,1,1.0,0.949054
11,2,0.0,0.929933


In [5]:
df_y_train = df_y_train.drop('proportion', axis=1)

In [7]:
df_y_train

,duration,event,proportion
0,2,0.0,0.226457
1,2,1.0,0.165919
6,3,0.0,0.552097
10,1,1.0,0.949054
11,2,0.0,0.929933
...,...,...,...
1896,3,1.0,0.177573
1898,0,1.0,0.795783
1901,0,1.0,0.887153
1902,3,0.0,0.253018


In [4]:
model = SurvTraceSingle(STConfig2)
trainer = Trainer(model)
train_loss, val_loss = trainer.fit((df_train, df_y_train), (df_val, df_y_val),
        batch_size=128,
        epochs=10,
        learning_rate=1e-3,
        weight_decay=1e-3,)

use pytorch-cuda for training.


b:\Python\surv_mlp\experiments\models\survtrace\train_utils.py:208: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha = 1) (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\python_arg_parser.cpp:1691.)
  next_m.mul_(beta1).add_(1 - beta1, grad)


[Train-0]: 17.120403289794922
[Val-0]: 1.6323128938674927
[Train-1]: 15.4667249917984
[Val-1]: 1.378175139427185
[Train-2]: 13.549633502960205
[Val-2]: 1.2347644567489624
[Train-3]: 11.623916506767273
[Val-3]: 1.2044042348861694
[Train-4]: 10.953188419342041
[Val-4]: 1.200553059577942
[Train-5]: 10.849285781383514
[Val-5]: 1.2418584823608398
EarlyStopping counter: 1 out of 5
[Train-6]: 10.832606852054596
[Val-6]: 1.1504303216934204
[Train-7]: 10.707978010177612
[Val-7]: 1.165506362915039
EarlyStopping counter: 1 out of 5
[Train-8]: 10.71961659193039
[Val-8]: 1.1640398502349854
EarlyStopping counter: 2 out of 5
[Train-9]: 10.560429096221924
[Val-9]: 1.1999845504760742
EarlyStopping counter: 3 out of 5


In [7]:
pred = model.predict_surv_df(df_test)

In [8]:
pred

,0,1,2,3,4,5,6,7,8,9,...,561,562,563,564,565,566,567,568,569,570
0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
42.683334,0.774157,0.955732,0.814162,0.857690,0.736108,0.978819,0.714174,0.952471,0.985579,0.970927,...,0.886781,0.806154,0.862645,0.727426,0.956071,0.916072,0.899766,0.805304,0.896716,0.955416
85.866669,0.606903,0.877513,0.644397,0.734989,0.553414,0.962861,0.608047,0.885643,0.974117,0.889652,...,0.644979,0.560709,0.734347,0.505073,0.855894,0.854380,0.838535,0.596787,0.758016,0.881418
145.333336,0.443554,0.659275,0.400543,0.361977,0.355154,0.838898,0.547854,0.654796,0.889343,0.602767,...,0.415177,0.406197,0.547150,0.234263,0.691596,0.776352,0.744187,0.439704,0.276333,0.643768
355.200012,0.317530,0.096454,0.047931,0.021869,0.072793,0.231019,0.322017,0.355581,0.534978,0.057135,...,0.053451,0.087776,0.205581,0.009484,0.140560,0.389414,0.523188,0.137939,0.010054,0.109484


In [10]:
df_train.head()

,x4,x5,x6,x7,x0,x1,x2,x3,x8
1,1.0,2.0,4.0,7.0,-1.073261,3.276337,-0.413222,-0.609223,1.915404
2,0.0,3.0,4.0,7.0,-0.336807,0.528473,1.227964,0.003054,-0.974702
3,0.0,2.0,4.0,6.0,0.513699,-0.877283,-1.561519,-0.635844,0.448772
4,1.0,2.0,4.0,7.0,-0.874052,-0.879345,-0.154487,0.397502,0.521217
5,1.0,3.0,4.0,7.0,-0.909893,0.092082,-0.228606,-0.362765,0.726992
